# How this was worked out

`demo.ipynb` shows that the client works. This one shows how it was arrived at,
following the three write-ups in `docs/`.

Every cell here runs against the capture files committed in `fixtures/`, so the
whole notebook is offline and deterministic. Nothing below asks a server for
anything, which means nothing below can be explained away by a good minute on
the network.

The order is the order it happened in: a capture, the question of which bytes
are allowed to move, the two extensions that took longest, and then the two
places where a second measurement contradicted the first.

In [ ]:
# This notebook lives in notebooks/ but every path below is relative to the
# repository root, and `import pybrws` needs the root importable. Resolving it
# here rather than hardcoding `..` means the notebook runs the same whether
# Jupyter was started in notebooks/ or at the root.
import os, sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "pybrws").is_dir())
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import sys
sys.path.insert(0, "tools")

from pybrws.differ import diff
from pybrws.emitter import build_client_hello
from pybrws.hexio import hexdump, load_hex
from pybrws.model import is_grease
from pybrws.parser import _Reader, parse_client_hello
from pybrws.profiles import chrome155, firefox157

import ja4

FIXTURES = {
    "tcp":      "fixtures/reference-firefox-157-windows.hex",
    "tcp2":     "fixtures/reference-firefox-157-windows-fresh.hex",
    "resumed":  "fixtures/reference-firefox-157-windows-resumed.hex",
    "quic":     "fixtures/reference-firefox-157-quic.hex",
}
raw = {k: load_hex(v) for k, v in FIXTURES.items()}
cap = {k: parse_client_hello(v) for k, v in raw.items()}


def rows(header, body, gap=2):
    """Fixed-width columns, which is all any table here needs."""
    table = [header] + [[str(c) for c in r] for r in body]
    width = [max(len(r[i]) for r in table) for i in range(len(header))]
    pad = " " * gap
    out = [pad.join(c.ljust(w) for c, w in zip(table[0], width)).rstrip(),
           pad.join("-" * w for w in width)]
    out += [pad.join(c.ljust(w) for c, w in zip(r, width)).rstrip() for r in table[1:]]
    print("\n".join(out))

---
## Part one — the capture

A browser's first packet, before any HTTP exists. There is no URL in it, no
`User-Agent`, no cookies: a list of cipher suites, a list of extensions in an
order, and key material.

In [ ]:
hello = cap["tcp"]
print(f"{len(raw['tcp'])} bytes\n")
print(hexdump(raw["tcp"][:80]))

In [ ]:
print(hello.summary())

### Which bytes am I allowed to change?

Before copying something you have to know which parts of it are *supposed* to
vary. Two captures taken under the same conditions, compared two ways: the
structural differ, which knows what is allowed to move, and a field-by-field
check of the four fields that must.

The differ returning nothing is the claim. The four `True`s underneath are what
stop that from being a trivial claim about two identical files.

In [ ]:
a, b = cap["tcp"], cap["tcp2"]

print("structural diff:", diff(a, b) or "[]  (nothing a server could key on)")
print()
rows(["field", "differs between the two captures", "length"],
     [["random",                 a.random != b.random,                               f"{len(a.random)}B, stable"],
      ["session_id",             a.session_id != b.session_id,                       f"{len(a.session_id)}B, stable"],
      ["key_share",              a.extension(51).data != b.extension(51).data,       f"{len(a.extension(51).data)}B, stable"],
      ["encrypted_client_hello", a.extension(65037).data != b.extension(65037).data, f"{len(a.extension(65037).data)}B, stable"],
      ["extension order",        a.extension_types() != b.extension_types(),         f"{len(a.extensions)} extensions"],
      ["cipher list",            a.cipher_suites != b.cipher_suites,                 f"{len(a.cipher_suites)} suites"]])

A fixed extension order was the best news in the project. It makes
*identical to the capture* a target you can hit, rather than a distribution you
have to imitate. Part two is about what happens when that stops being true.

### The 1,327-byte extension

Seventy percent of the message is `key_share`. The first attempt at it was to
paste the captured bytes in, which fails at the end of the handshake with no
error naming the cause: a `key_share` is the client's half of a key exchange,
so sending a captured one means sending a public key whose private half you do
not have.

The big entry is post-quantum. 1,216 is 1184 + 32 — an ML-KEM-768 key, then an
X25519 key, in that order.

In [ ]:
def entries(ext):
    reader = _Reader(_Reader(ext.data).vector(2))
    while reader.remaining:
        group = reader.uint(2)
        yield group, reader.vector(2)

NAMES = {0x11ec: "X25519MLKEM768", 0x001d: "X25519", 0x0017: "secp256r1",
         0x0018: "secp384r1", 0x0019: "secp521r1"}

sent = [(g, len(s)) for g, s in entries(hello.extension(51))]
rows(["group", "name", "bytes"],
     [[f"0x{g:04x}", NAMES.get(g, "?"), n] for g, n in sent])

groups = _Reader(_Reader(hello.extension(10).data).vector(2))
advertised = []
while groups.remaining:
    advertised.append(groups.uint(2))

print(f"\nadvertised in supported_groups : {len(advertised)}  "
      + " ".join(NAMES.get(g, hex(g)) for g in advertised))
print(f"shares actually sent           : {len(sent)}")
print("\nThe asymmetry is not a bug. A server may answer HelloRetryRequest asking")
print("for any advertised group, so all five have to be producible. It is also")
print("part of the fingerprint: 5 advertised and 3 sent is a specific shape.")

### The extension I assumed was noise

Extension 65037 is 282 bytes that change completely between captures. For a day
it looked like padding. It is `encrypted_client_hello`, and decoding it across
captures is what turns the plan from *hardcode these 282 bytes* into something
that survives being used twice.

The plan would have produced a client that is perfectly correct on one
connection and obvious over a hundred, because a real Firefox varies `aead_id`
and `config_id` and a hardcoded copy varies nothing.

In [ ]:
def ech(hello):
    reader = _Reader(hello.extension(65037).data)
    return {"type": reader.uint(1), "kdf_id": reader.uint(2),
            "aead_id": reader.uint(2), "config_id": reader.uint(1),
            "enc": len(reader.vector(2)), "payload": len(reader.vector(2))}

fields = ["type", "kdf_id", "aead_id", "config_id", "enc", "payload"]
one, two = ech(cap["tcp"]), ech(cap["tcp2"])
rows(["field", "capture 1", "capture 2", "across captures"],
     [[f, f"0x{one[f]:04x}" if f.endswith("id") else one[f],
          f"0x{two[f]:04x}" if f.endswith("id") else two[f],
          "varies" if one[f] != two[f] else "constant"] for f in fields])
print("\nFirefox sends this whether or not it is doing real ECH: if only ECH users")
print("sent the extension, sending it would itself be the signal. This is the decoy.")

### Byte identity

The strongest form of the claim. Feed the emitter the capture's *own* random
material — its random, its session id, its ECH draws — and its own key_share
entries, and ask whether what comes out is the capture.

Not a hash matching. Not a field-by-field review. The 1,901 bytes.

In [ ]:
from tests.test_emitter import REFERENCE_HOSTNAME, ReplayKeyExchange, replay_script

queue = replay_script(cap["tcp"])

def replay(n):
    chunk = queue.pop(0)
    assert len(chunk) == n, f"emitter asked for {n}, script holds {len(chunk)}"
    return chunk

built = build_client_hello(REFERENCE_HOSTNAME, rng=replay,
                           key_exchange=ReplayKeyExchange(cap["tcp"]))

print("built   ", len(built), "bytes")
print("capture ", len(raw["tcp"]), "bytes")
print("identical:", built == raw["tcp"])
print("random left unconsumed:", queue or "none  (the emitter asked for exactly what the capture holds)")

That last line matters as much as the comparison. An emitter that drew *less*
randomness than the capture carries would still come out byte-identical if the
leftovers happened to sit at the end, and would be wrong about where the
variability lives.

### JA4, computed here rather than quoted

`tools/ja4.py` builds the fingerprint from a parsed hello, so the published
value can be recomputed from the committed capture instead of trusted.

In [ ]:
_ja4=ja4.compute(cap["tcp"], set())["ja4"]
print(_ja4)
print()
print(f"{_ja4.split("_")[0]}")
print("│  │ │ │ └─ first and last char of the first ALPN entry")
print("│  │ │ └─── extension count")
print("│  │ └───── cipher count")
print("│  └──────── d: SNI present")
print("└─────────── TLS 1.3 over TCP")

---
## Part two — Chrome, which never sends the same thing twice

The Firefox profile is a list of constants. Six captures of Chrome produced six
different extension orders, which means there is no order to store, and the
profile format that worked for one browser was never a format at all — it was
one browser written down.

Below, each profile is asked to build six hellos and the orders are counted.

In [ ]:
def orders(profile, n=6):
    seen = []
    for _ in range(n):
        parsed = parse_client_hello(build_client_hello("example.com", profile=profile))
        seen.append(tuple("G" if is_grease(t) else t for t in parsed.extension_types()))
    return seen

for profile in (firefox157, chrome155):
    seen = orders(profile)
    print(f"{profile.NAME:12s} {len(set(seen))} distinct order(s) in {len(seen)} connections")

In [ ]:
# The same question, printed. 'G' is a GREASE value, which is drawn per connection.
for profile in (firefox157, chrome155):
    print(profile.NAME)
    for order in orders(profile, 4):
        print("   ", " ".join(str(t) for t in order))
    print()

### The two that never move

Positions one and last never change, and what sits in them are not real
extensions. GREASE (RFC 8701) is sixteen reserved values that clients send on
purpose, as garbage, so that a middlebox which hardcodes *I know every valid
extension* is discovered continuously rather than on the day a real new
extension ships.

Chrome puts one at each end — exactly where a lazy parser is most likely to be
special-cased. Firefox sends none at all. Two browsers, opposite policies, both
conformant, which is the whole reason fingerprinting works.

In [ ]:
parsed = parse_client_hello(build_client_hello("example.com", profile=chrome155))
types = parsed.extension_types()
print(f"chrome155: {len(types)} extensions")
print(f"  first  0x{types[0]:04x}  GREASE: {is_grease(types[0])}")
print(f"  last   0x{types[-1]:04x}  GREASE: {is_grease(types[-1])}")

ff = parse_client_hello(build_client_hello("example.com", profile=firefox157))
print(f"\nfirefox157: {sum(is_grease(t) for t in ff.extension_types())} GREASE extensions")

### The bug that happens one time in sixteen

Drawing the two pinned values independently is fine fifteen times out of
sixteen. On the sixteenth both draws return the same value, and the hello now
carries two extensions of the same type, which RFC 8446 forbids outright.

A fingerprinting service will not reject it. It will record a client no browser
produces — and the quieter tell is that a duplicate makes the hello one byte
shorter than any real capture. That is the kind of thing you ship forever if
you are eyeballing diffs instead of asserting lengths.

It is bounded rejection sampling now. The cell is the regression test: 2,000
connections, and the pair is never equal.

In [ ]:
lengths, collisions = set(), 0
for _ in range(2000):
    message = build_client_hello("example.com", profile=chrome155)
    types = parse_client_hello(message).extension_types()
    lengths.add(len(message))
    collisions += types[0] == types[-1]

print(f"2000 hellos, pinned pair equal: {collisions}")
print(f"byte lengths produced: {sorted(lengths)}")
print()
print("Several lengths is correct for Chrome and would be a defect in Firefox,")
print("whose capture is 1,901 bytes every time:")
print("  firefox157", sorted({len(build_client_hello(REFERENCE_HOSTNAME, profile=firefox157))
                              for _ in range(50)}), "  (the reference hostname; a shorter name shifts it)")

### The field trial

Chrome's published capture sets disagree with this profile by one extension,
`0x12e0`. The first reading was that the profile was broken. The second was
that headless Chrome omits it, and that is a tell you can read with no
JavaScript anywhere.

Both were wrong. `0x12e0` is gated behind a randomised field trial that one
install joins and a fresh profile does not — and a capture set built from
containers is systematically the control arm.

Dropping it from the computation lands on the catalogue's value exactly, which
is what makes the explanation a measurement rather than a story.

In [ ]:
mine = parse_client_hello(build_client_hello("example.com", profile=chrome155))
rows(["computation", "ja4"],
     [["this profile, as captured", ja4.compute(mine, set())["ja4"]],
      ["with 0x12e0 dropped",       ja4.compute(mine, {0x12e0})["ja4"]]])
print("\nThe second value is the one in thumbprint.me's catalogue for Chrome 155.")
print("Both are real Chrome. 'Chrome 155's JA4' is not a value, it is a")
print("distribution over trial arms, and the seed rotates on Google's schedule")
print("rather than Chrome's release schedule.")

---
## Part three — one browser, two fingerprints

A client that speaks HTTP/2 while claiming to be a browser that prefers HTTP/3
announces itself as loudly as one with the wrong cipher list. So the same
Firefox 157 was captured over QUIC.

QUIC carries the TLS handshake inside packets protected by keys derived from a
salt printed in RFC 9001 — a fixed constant anyone can compute — so the
ClientHello is as readable there as over TCP, with no server needed. The
fixture below came out of a capture of ordinary traffic.

In [ ]:
rows(["capture", "bytes", "ciphers", "extensions", "ja4"],
     [[name, len(raw[key]), len(cap[key].cipher_suites), len(cap[key].extensions),
       ja4.compute(cap[key], set())["ja4"]]
      for name, key in [("TCP, fresh", "tcp"), ("TCP, resumed", "resumed"),
                        ("QUIC", "quic")]])
print("\nOne caveat about the instrument: tools/ja4.py pins TRANSPORT to 't', so")
print("the leading letter of the QUIC row says TCP. JA4 specifies 'q' there. The")
print("rest of the string is computed from the hello and stands; the first")
print("character is the tool's, not the capture's.")

In [ ]:
over_tcp = set(cap["tcp"].extension_types())
over_quic = set(cap["quic"].extension_types())

from pybrws.model import extension_name
print("TCP only :", ", ".join(sorted(extension_name(t) for t in over_tcp - over_quic)))
print("QUIC only:", ", ".join(sorted(extension_name(t) for t in over_quic - over_tcp)))
print("\nSame browser, same version, same machine. Firefox's QUIC stack is neqo,")
print("in Rust; its TCP stack is NSS. Two codebases, two sets of habits.")

Three fingerprints from one browser, and the resumed one is worth its own line:
a resumed hello carries `pre_shared_key`, so a client that resumes presents a
second JA4 that no profile built from a fresh capture accounts for. That is
listed as an open gap in the README rather than solved here.

What this notebook does **not** show is the shuffle finding from part three —
that Firefox draws a fresh extension order per connection over QUIC while
keeping a stored one over TCP. One QUIC capture is committed and that claim
needs six, so it stays in the post with its evidence rather than being restated
here with less.

---
## The limit

None of this touches a JavaScript challenge. An interstitial that runs a script
and waits for an answer does not care about the handshake — it challenges a real
Chrome too, and a real Chrome passes with a capability, not a disguise.

The measured gap that follows from that is in `docs/chrome.html`: yelp serves
this client's Firefox profile and refuses its Chrome profile, and so does
`curl_cffi`'s Chromium, because the site challenges Chromium-shaped clients. The
profile is not the problem and a better profile is not the fix.

`demo.ipynb` has the network-facing side: which targets this changes, and the
measurement showing the fingerprint is necessary and not sufficient.